In [5]:
!pip uninstall -y massi
!pip cache purge

Found existing installation: massi 0.1.0
Uninstalling massi-0.1.0:
  Successfully uninstalled massi-0.1.0
Files removed: 685 (66.7 MB)


In [6]:
# !cat ../massi/agents/massi.py
!cd .. && pip install -v . --group=dev --group=observability

Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
Processing /home/jovyan/work
  Running command installing build dependencies
  Using pip 25.3 from /opt/conda/lib/python3.13/site-packages/pip (python 3.13)
    Obtaining dependency information for setuptools>=40.8.0 from https://files.pythonhosted.org/packages/e1/c6/76dc613121b793286a3f91621d7b75a2b493e0390ddca50f11993eadf192/setuptools-82.0.0-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.0 MB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 7.5 MB/s  0:00:00
  Installing build dependencies ... done
  Running command Getting requirements to build wheel
  running egg_info
  writing massi.egg-info/PKG-INFO
  writing dependency_links to massi.egg-info/dependency_links.txt
  writing requirements to massi.egg-info/requires.txt
  writing top-level names to massi.egg-info/top_level.txt
  reading manifest file 'massi.egg-info/SOURCES.txt'
  writing manifest file 

In [7]:
# Setup observability
import os
from lmnr import Laminar

Laminar.initialize(
    project_api_key=os.environ["LMNR_PROJECT_API_KEY"],
    base_url=f"http://{os.environ["HOST_MACHINE_URL_CORE"]}",
    http_port=int(os.environ["LMNR_PROJECT_HTTP_PORT"]),
    grpc_port=int(os.environ["LMNR_PROJECT_GRPC_PORT"]),
)

1. **We will use the relay model for the sake of debugging convenience**

In [8]:
from massi.agents.massi import MassiScenarioFactory, Config, Dependencies, StandardIOPromptClient

In [9]:
from massi.agents.stunts import RelayChatModel
relay_model = RelayChatModel()

In [10]:
config = Config(**{"model_url": "http://host.docker.internal:11434", "model_name": "granite4:3b"})
deps = Dependencies(llm_client=RelayChatModel(), prompt_client=StandardIOPromptClient())

In [12]:
# OpenAI model usage
from massi.agents.massi import create_openai_llm

oa_config = Config(**{"api_key": os.environ["OPENAI_API_TOKEN"], "model_name": "gpt-5-nano"})
ao_deps = Dependencies(llm_client=create_openai_llm(oa_config), prompt_client=StandardIOPromptClient())

In [13]:
factory = MassiScenarioFactory(config)

In [29]:
USE_MOCK_DEPS = False
USE_OPENAI_MODELS = False

if USE_MOCK_DEPS:
    print("... using mocking dependencies ...")
    graph = factory.resource_property_graph(deps)
elif USE_OPENAI_MODELS:
    print("... using OpenAI dependencies ...")
    graph = factory.resource_property_graph(ao_deps)
else:
    print("... using standard dependencies ...")
    graph = factory.resource_property_graph()

... using standard dependencies ...


In [30]:
import grandalf

graph.get_graph().print_ascii()

   +-----------+     
   | __start__ |     
   +-----------+     
          *          
          *          
          *          
+----------------+   
| scan_directory |   
+----------------+   
          .          
          .          
          .          
+-----------------+  
| score_relevancy |  
+-----------------+  
          *          
          *          
          *          
+-----------------+  
| group_resources |  
+-----------------+  
          *          
          *          
          *          
    +---------+      
    | __end__ |      
    +---------+      


In [31]:
import random

thread = {"configurable": {"thread_id": str(random.randint(1, 100))}}

In [32]:
import pprint

In [33]:
def stream_until_iterruption(graph, *args):
    for event in graph.stream(*args, stream_mode="values"):
        print("\n==== new event ====")
        for k, v in event.items():
            print(f"---- {k} ----")
            pprint.pp(v, indent=4)

In [34]:
DEFAULT_SCORING_RESPONSE = """
{ "resources": [
    {"file_location": ".DS_Store", "filename": ".DS_Store", "extension": "", "author": "jovyan", "gis_relevancy_score": 0.0},
    {"file_location": "uc1-massi_demo--missing_data.ipynb", "filename": "uc1-massi_demo--missing_data.ipynb", "extension": ".ipynb", "author": "jovyan", "gis_relevancy_score": 0.8},
    {"file_location": "base_metadata.json", "filename": "base_metadata.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "uc2-massi_demo--resource_inference.ipynb", "filename": "uc2-massi_demo--resource_inference.ipynb", "extension": ".ipynb", "author": "jovyan", "gis_relevancy_score": 0.8},
    {"file_location": "massi_demo--print_ascii-test.ipynb", "filename": "massi_demo--print_ascii-test.ipynb", "extension": ".ipynb", "author": "jovyan", "gis_relevancy_score": 0.3},
    {"file_location": ".ipynb_checkpoints/uc1-massi_demo--missing_data-checkpoint.ipynb", "filename": "uc1-massi_demo--missing_data-checkpoint.ipynb", "extension": ".ipynb", "author": "jovyan", "gis_relevancy_score": 0.1},
    {"file_location": ".ipynb_checkpoints/base_metadata-checkpoint.json", "filename": "base_metadata-checkpoint.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.1},
    {"file_location": ".ipynb_checkpoints/uc2-massi_demo--resource_inference-checkpoint.ipynb", "filename": "uc2-massi_demo--resource_inference-checkpoint.ipynb", "extension": ".ipynb", "author": "jovyan", "gis_relevancy_score": 0.1},
    {"file_location": ".ipynb_checkpoints/massi_demo--print_ascii-test-checkpoint.ipynb", "filename": "massi_demo--print_ascii-test-checkpoint.ipynb", "extension": ".ipynb", "author": "jovyan", "gis_relevancy_score": 0.1},
    {"file_location": "data/.DS_Store", "filename": ".DS_Store", "extension": "", "author": "jovyan", "gis_relevancy_score": 0.0},
    {"file_location": "data/layer_data/TRI__IA-LA.geojson", "filename": "TRI__IA-LA.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/gw_sampled_and_detected.json", "filename": "gw_sampled_and_detected.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/groundwater_not_detected.geojson", "filename": "groundwater_not_detected.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/Model_says_NO.geojson", "filename": "Model_says_NO.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/TRI_CA-HI.geojson", "filename": "TRI_CA-HI.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/TRI__NC-OH.geojson", "filename": "TRI__NC-OH.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/WATER_SYSTEMS_IN_KENTUCKY.geojson", "filename": "WATER_SYSTEMS_IN_KENTUCKY.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/hot-spot_map.csv", "filename": "hot-spot_map.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_2.geojson", "filename": "Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_2.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/groundwater_not_detected.json", "filename": "groundwater_not_detected.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI__MA-MT.csv", "filename": "TRI__MA-MT.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI_CA-HI.csv", "filename": "TRI_CA-HI.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/PFAS_detected_sites.geojson", "filename": "PFAS_detected_sites.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/Model_says_NO.csv", "filename": "Model_says_NO.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI_DATA_IN_KENTUCKY.json", "filename": "TRI_DATA_IN_KENTUCKY.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Model_says_NO.json", "filename": "Model_says_NO.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/water_district.json", "filename": "water_district.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3.geojson", "filename": "Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/PFAS_sampling_and_purchasing_and_intake_detail.geojson", "filename": "PFAS_sampling_and_purchasing_and_intake_detail.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/waterIntake.json", "filename": "waterIntake.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI__TX-WV.geojson", "filename": "TRI__TX-WV.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/TRI_only_A.geojson", "filename": "TRI_only_A.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/water_district.csv", "filename": "water_district.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI_only_A.json", "filename": "TRI_only_A.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_1.geojson", "filename": "Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_1.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/TRI_DATA_IN_KENTUCKY.geojson", "filename": "TRI_DATA_IN_KENTUCKY.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/TRI__IA-LA.json", "filename": "TRI__IA-LA.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI_CA-HI.json", "filename": "TRI_CA-HI.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Model_says_yes.csv", "filename": "Model_says_yes.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_2.json", "filename": "Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_2.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Model_says_yes.geojson", "filename": "Model_says_yes.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/hot-spot_map.geojson", "filename": "hot-spot_map.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/water_district.geojson", "filename": "water_district.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/WATER_SYSTEMS_IN_KENTUCKY.csv", "filename": "WATER_SYSTEMS_IN_KENTUCKY.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Surfacewater_sampled_and_detected.json", "filename": "Surfacewater_sampled_and_detected.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI__NC-OH.csv", "filename": "TRI__NC-OH.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3.json", "filename": "Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/PFAS_sampling_and_purchasing_and_intake_detail.csv", "filename": "PFAS_sampling_and_purchasing_and_intake_detail.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI_only_A.csv", "filename": "TRI_only_A.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Risk_map_with_landfill.csv", "filename": "Risk_map_with_landfill.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/PFAS_sampling_and_purchasing_and_intake_detail.json", "filename": "PFAS_sampling_and_purchasing_and_intake_detail.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_1.csv", "filename": "Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_1.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Risk_map_with_landfill.geojson", "filename": "Risk_map_with_landfill.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/TRI__MA-MT.geojson", "filename": "TRI__MA-MT.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/Surfacewater_sampled_and_detected.csv", "filename": "Surfacewater_sampled_and_detected.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_2.csv", "filename": "Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_2.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI__TX-WV.csv", "filename": "TRI__TX-WV.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Model_says_yes.json", "filename": "Model_says_yes.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/groundwater_not_detected.csv", "filename": "groundwater_not_detected.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3.csv", "filename": "Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_3.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_1.json", "filename": "Ky_Water_Resources_Polygons_DOW_SWAPP_Zone_1.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/waterIntake.csv", "filename": "waterIntake.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI__NC-OH.json", "filename": "TRI__NC-OH.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/waterIntake.geojson", "filename": "waterIntake.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/TRI__OK-TN.csv", "filename": "TRI__OK-TN.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI__OK-TN.geojson", "filename": "TRI__OK-TN.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/Surfacewater_sampled_and_detected.geojson", "filename": "Surfacewater_sampled_and_detected.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/WATER_SYSTEMS_IN_KENTUCKY.json", "filename": "WATER_SYSTEMS_IN_KENTUCKY.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI_DATA_IN_KENTUCKY.csv", "filename": "TRI_DATA_IN_KENTUCKY.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/hot-spot_map.json", "filename": "hot-spot_map.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Surface_water_sampled_and_not_detected.csv", "filename": "Surface_water_sampled_and_not_detected.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/gw_sampled_and_detected.geojson", "filename": "gw_sampled_and_detected.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/layer_data/gw_sampled_and_detected.csv", "filename": "gw_sampled_and_detected.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI__OK-TN.json", "filename": "TRI__OK-TN.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI__MA-MT.json", "filename": "TRI__MA-MT.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Risk_map_with_landfill.json", "filename": "Risk_map_with_landfill.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI__IA-LA.csv", "filename": "TRI__IA-LA.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/TRI__TX-WV.json", "filename": "TRI__TX-WV.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Surface_water_sampled_and_not_detected.json", "filename": "Surface_water_sampled_and_not_detected.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/PFAS_detected_sites.json", "filename": "PFAS_detected_sites.json", "extension": ".json", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/PFAS_detected_sites.csv", "filename": "PFAS_detected_sites.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/layer_data/Surface_water_sampled_and_not_detected.geojson", "filename": "Surface_water_sampled_and_not_detected.geojson", "extension": ".geojson", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/other/gis_distance_matrix.py", "filename": "gis_distance_matrix.py", "extension": ".py", "author": "jovyan", "gis_relevancy_score": 1.0},
    {"file_location": "data/other/SUPERFUND SITES RAW.xlsx", "filename": "SUPERFUND SITES RAW.xlsx", "extension": ".xlsx", "author": "jovyan", "gis_relevancy_score": 0.7},
    {"file_location": "data/other/HISTORIC LANDFILLS MAP- 2019 RAW.csv", "filename": "HISTORIC LANDFILLS MAP- 2019 RAW.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.8},
    {"file_location": "data/other/AI LIST REPORTS RAW.csv", "filename": "AI LIST REPORTS RAW.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.6},
    {"file_location": "data/other/PFAS DRINKING WATER REPORT FINAL.pdf", "filename": "PFAS DRINKING WATER REPORT FINAL.pdf", "extension": ".pdf", "author": "jovyan", "gis_relevancy_score": 0.5},
    {"file_location": "data/other/PFA_SITES.csv", "filename": "PFA_SITES.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/other/Model result for regression .csv", "filename": "Model result for regression .csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.7},
    {"file_location": "data/other/all_12_mile_water_risk_TRI.csv", "filename": "all_12_mile_water_risk_TRI.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/other/KY_WATER_SYSTEMS.csv", "filename": "KY_WATER_SYSTEMS.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.9},
    {"file_location": "data/other/TRI_2019_KY RAW.csv", "filename": "TRI_2019_KY RAW.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.8},
    {"file_location": "data/other/TRI_2019_US RAW.csv", "filename": "TRI_2019_US RAW.csv", "extension": ".csv", "author": "jovyan", "gis_relevancy_score": 0.8}
  ]
}
"""

In [35]:
from unittest.mock import patch

MOCK_RUN = False

In [ ]:
# Run the graph until the first interruption
if MOCK_RUN:
    print("... streaming with a mock pass ...\n")
    with patch('builtins.input', return_value=DEFAULT_SCORING_RESPONSE):
        stream_until_iterruption(graph, {}, thread)
else: 
    print("... streaming with a standard pass ...\n")
    stream_until_iterruption(graph, {}, thread)

... streaming with a standard pass ...


==== new event ====
---- messages ----
[]
---- files ----
[   ResourceFile(file_location='.DS_Store', filename='.DS_Store', extension='', author='jovyan'),
    ResourceFile(file_location='uc1-massi_demo--missing_data.ipynb', filename='uc1-massi_demo--missing_data.ipynb', extension='.ipynb', author='jovyan'),
    ResourceFile(file_location='base_metadata.json', filename='base_metadata.json', extension='.json', author='jovyan'),
    ResourceFile(file_location='uc2-massi_demo--resource_inference.ipynb', filename='uc2-massi_demo--resource_inference.ipynb', extension='.ipynb', author='jovyan'),
    ResourceFile(file_location='massi_demo--observation-sandbox.ipynb', filename='massi_demo--observation-sandbox.ipynb', extension='.ipynb', author='jovyan'),
    ResourceFile(file_location='.ipynb_checkpoints/uc1-massi_demo--missing_data-checkpoint.ipynb', filename='uc1-massi_demo--missing_data-checkpoint.ipynb', extension='.ipynb', author='jovyan'),
    Res

In [24]:
type(graph.get_state(thread).values["gis_scored_files"])

list

In [25]:
graph.get_state(thread).values["gis_scored_files"]

[GISScoredFile(file_location='.DS_Store', filename='.DS_Store', extension='', author='jovyan', gis_relevancy_score=0.0),
 GISScoredFile(file_location='uc1-massi_demo--missing_data.ipynb', filename='uc1-massi_demo--missing_data.ipynb', extension='.ipynb', author='jovyan', gis_relevancy_score=0.4),
 GISScoredFile(file_location='base_metadata.json', filename='base_metadata.json', extension='.json', author='jovyan', gis_relevancy_score=0.75),
 GISScoredFile(file_location='uc2-massi_demo--resource_inference.ipynb', filename='uc2-massi_demo--resource_inference.ipynb', extension='.ipynb', author='jovyan', gis_relevancy_score=0.4),
 GISScoredFile(file_location='massi_demo--observation-sandbox.ipynb', filename='massi_demo--observation-sandbox.ipynb', extension='.ipynb', author='jovyan', gis_relevancy_score=0.5),
 GISScoredFile(file_location='.ipynb_checkpoints/uc1-massi_demo--missing_data-checkpoint.ipynb', filename='uc1-massi_demo--missing_data-checkpoint.ipynb', extension='.ipynb', author='jo

In [28]:
graph.get_state(thread).values["gis_resource_groups"]

[GisResourceGroup(resource_name='', entries=[GISScoredFile(file_location='.DS_Store', filename='.DS_Store', extension='', author='jovyan', gis_relevancy_score=0.0), GISScoredFile(file_location='data/.DS_Store', filename='.DS_Store', extension='', author='jovyan', gis_relevancy_score=0.0)]),
 GisResourceGroup(resource_name='uc1-massi_demo--missing_data', entries=[GISScoredFile(file_location='uc1-massi_demo--missing_data.ipynb', filename='uc1-massi_demo--missing_data.ipynb', extension='.ipynb', author='jovyan', gis_relevancy_score=0.4)]),
 GisResourceGroup(resource_name='base_metadata', entries=[GISScoredFile(file_location='base_metadata.json', filename='base_metadata.json', extension='.json', author='jovyan', gis_relevancy_score=0.75)]),
 GisResourceGroup(resource_name='uc2-massi_demo--resource_inference', entries=[GISScoredFile(file_location='uc2-massi_demo--resource_inference.ipynb', filename='uc2-massi_demo--resource_inference.ipynb', extension='.ipynb', author='jovyan', gis_relevanc